# Практическая работа №1. Часть 4Г
## Языковые модели на трансформере: свой маленький GPT, кодировщик BERT и его дообучение

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ExPl0i/AI-10-NLP-Language-models/blob/main/%D0%9F%D0%A0-%D0%AF%D0%B7%D1%8B%D0%BA%D0%BE%D0%B2%D1%8B%D0%B5%20%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8.ipynb)

**Цель:** собрать из блоков прошлого занятия модель, которая пишет текст (свой маленький GPT на своем токенизаторе BPE), измерить ее качество перплексией и сравнить с символьной рекуррентной моделью; затем взять готовый кодировщик rubert-tiny2, построить векторы фраз водителя для смыслового поиска и дообучить его определять намерение водителя. Дообученный классификатор понадобится на занятии Т2.Б2.5, где соберем голосового помощника.

> **Правило части.** Модель GPT, ее обучение, генерацию, перплексию, векторы фраз, смысловой поиск и метрики классификатора пишем сами. Готовые блоки трансформера (внимание, маска, многоголовое внимание, блок) берем из Т2.Б1.3, токенизатор BPE – из Т2.Б1.1. Библиотека `transformers` нужна только для загрузки rubert-tiny2.

**Данные.**
- Корпус русской классики (`data/corpus.txt`) и правила BPE из Т2.Б1.1 (`data/bpe_light.json` – словарь на 1000 токенов, `data/bpe_full.json` – на 2000). Корпус в токенах: 4 366 018 токенов в легком режиме и 3 976 118 в полном; последние 5 % (Чехов) – проверочная часть.
- Фразы водителя с пятью намерениями: навигация, климат, музыка, звонок, состояние машины. Обучающие фразы – ваш файл `intents.csv` из домашнего задания Т2.Б1.3 (столбцы `text` и `intent`); если его нет рядом с ноутбуком, берется запасной набор `data/intents_reserve.csv` (1000 фраз, по 200 на намерение, собраны по шаблонам). Тестовые фразы `data/intents_test.csv` – 100 живых фраз, написанных вручную другим стилем («Что-то я замерз», «Мигает какой-то значок на панели»): на них видно, понимает ли модель смысл, а не шаблон.

**Два режима** выбираются в первой ячейке и меняют только GPT (задания Г1–Г2); rubert-tiny2 в обоих режимах работает на процессоре.
- `'легкий'` – для компьютера в аудитории: GPT около 0,9 млн параметров, словарь BPE на 1000 токенов, окно 64 токена, 1500 шагов на процессоре (несколько минут);
- `'полный'` – для Colab с видеокартой: GPT около 11 млн параметров, словарь на 2000 токенов, окно 256 токенов, 2000 шагов.

Обучение GPT можно пропустить: `SKIP_TRAINING = True` загружает готовую контрольную точку (`data/gpt_light.npz` для легкого режима; для полного – файл `gpt_full.pt`, если преподаватель его выложил). Во время обучения контрольная точка сохраняется каждые 250 шагов, и после отключения сеанса обучение продолжается с нее; в Colab можно сохранять ее на Google Диск (`USE_DRIVE = True`).

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка в комментарии `# <--`. Проверки на маленьких примерах должны совпасть с тем, что стоит после слова «ожидается», точно. Результаты обучения зависят от случайных чисел, поэтому для них ожидания даны с запасом.

## Как из блоков получаются языковые модели

**1. Три семейства трансформеров.** На прошлом занятии мы собрали блок трансформера. Из одинаковых блоков строят три вида моделей:

| Семейство | Маска | Чему учится | Примеры | Для чего |
| --- | --- | --- | --- | --- |
| только декодер | причинная | предсказывать следующий токен | GPT, Qwen, LLaMA | писать текст слово за словом |
| только кодировщик | нет | угадывать закрытые маской токены по словам слева и справа | BERT, rubert-tiny2 | понимать текст целиком: классификация, поиск |
| кодировщик и декодер | нет в кодировщике, причинная в декодере | переводить одну последовательность в другую | T5, Whisper | перевод, распознавание речи, пересказ |

У кодировщика с декодером в декодере есть еще **перекрестное внимание**: запросы берутся из декодера, а ключи и значения – из выхода кодировщика. Так декодер Whisper, когда пишет очередное слово, смотрит на нужный кусок звука (Т2.Б2.2).

**2. Как учится модель-декодер.** Текст из $T$ токенов $t_1, \dots, t_T$ дает сразу $T - 1$ обучающих примеров: по токенам $t_1, \dots, t_i$ предсказать $t_{i+1}$. Причинная маска позволяет посчитать все предсказания за один проход: позиция $i$ видит только токены до себя, а ответ для нее – следующий токен. Вход модели – отрезок текста, цель – тот же отрезок, сдвинутый на один токен вправо. Функция потерь – средняя перекрестная энтропия:
$$L = -\frac{1}{T-1}\sum_{i=1}^{T-1} \log p(t_{i+1} \mid t_1, \dots, t_i).$$

**3. Перплексия.** Качество языковой модели измеряют **перплексией** на проверочном тексте: $\mathrm{PPL} = e^{L}$. Ее смысл: модель так же неуверена, как если бы на каждом шаге выбирала равновероятно из $\mathrm{PPL}$ токенов. Модель, которая ничего не знает и выбирает любой из $V$ токенов словаря равновероятно, имеет перплексию $V$, а потери $\ln V$; поэтому в начале обучения потери около $\ln V$ ($\ln 1000 \approx 6{,}91$). Перплексии моделей с разными токенизаторами сравнивать нельзя: токены разной длины. Для сравнения пересчитывают потери в **биты на символ**:
$$\mathrm{BPC} = \frac{L \cdot N_{\text{токенов}}}{N_{\text{символов}} \cdot \ln 2},$$
то есть сколько бит нужно в среднем, чтобы закодировать один символ текста с помощью модели. Так можно сравнить наш GPT на токенах BPE с символьной рекуррентной моделью из Темы 1.

**4. Генерация.** Модель-декодер пишет текст по одному токену: подать начало текста, взять распределение для последней позиции, выбрать токен, дописать его в конец и повторить. Окно контекста ограничено, поэтому в модель подаются только последние `ctx` токенов. Выбирать можно самый вероятный токен или случайный по распределению; температура $\tau$ делит оценки перед softmax: при $\tau < 1$ модель осторожнее, при $\tau > 1$ смелее. Подробно способы выбора разберем на Т2.Б1.5.

**5. Как учится модель-кодировщик.** BERT (Девлин и соавторы, 2019) учится **маскированному языковому моделированию**: в тексте случайно выбирают 15 % токенов, из них 80 % заменяют специальным токеном `[MASK]`, 10 % – случайным токеном, 10 % оставляют как есть, и модель угадывает исходные токены по словам слева и справа. Маски внимания нет, каждое слово видит всю фразу, поэтому кодировщик хорошо понимает текст, но не умеет писать его слово за словом. В начало каждого текста ставится служебный токен `[CLS]`; его выходной вектор часто используют как вектор всей фразы.

**6. Дообучение.** Обучать кодировщик с нуля дорого: нужны миллиарды слов. Поэтому берут готовый и **дообучают**: сверху добавляют маленькую голову (слой `nn.Linear` из вектора `[CLS]` в число классов) и немного учат всю модель на своей задаче с маленьким шагом обучения ($10^{-5}$–$10^{-4}$) в течение нескольких эпох. Это то же трансферное обучение, что в Теме 1 (Б2.5) для сверточных сетей: знания о языке берутся из предобучения, а голова учится различать наши классы. Можно заморозить кодировщик и учить только голову: быстрее, но обычно хуже.

**7. rubert-tiny2.** Маленький кодировщик BERT для русского языка с Hugging Face (`cointegrated/rubert-tiny2`): около 29 миллионов параметров, вектор токена из 312 чисел, словарь WordPiece на 83 828 токенов, лицензия MIT. Модель дообучена давать осмысленные векторы предложений: по ее карточке вектором фразы служит выход токена `[CLS]`, нормированный к длине 1. Близость фраз – косинус, как у векторов слов (Т2.Б1.2), но теперь вектор строится с учетом всей фразы и порядка слов.

**8. Смысловой поиск.** Все фразы базы один раз превращают в векторы; запрос тоже превращают в вектор и ищут фразы с наибольшим косинусом. Такой поиск находит «мне холодно» по фразе «сделай теплее», хотя у них нет общих слов. На этом же принципе работает поиск по документам для больших языковых моделей (Т2.Б1.5).

**9. Метрики классификатора** (9 класс, метрики качества). **Матрица ошибок** $C$: строка – истинный класс, столбец – предсказанный, $C_{ij}$ – сколько примеров класса $i$ модель отнесла к классу $j$. Для каждого класса $k$:
$$\mathrm{precision}_k = \frac{C_{kk}}{\sum_i C_{ik}},\quad \mathrm{recall}_k = \frac{C_{kk}}{\sum_j C_{kj}},\quad F_k = \frac{2\,\mathrm{precision}_k\,\mathrm{recall}_k}{\mathrm{precision}_k + \mathrm{recall}_k}.$$
**Макро-F-мера** – среднее $F_k$ по классам: каждый класс одинаково важен, даже если примеров в нем мало.

**10. В PyTorch и transformers.**
- `F.cross_entropy(logits.view(-1, V), targets.view(-1))` – потери по всем позициям сразу; `torch.multinomial(probs, 1)` – случайный выбор по распределению;
- `model.head.weight = model.tok.weight` – общие веса входных эмбеддингов и выходного слоя (так делают исходный трансформер и GPT-2: меньше параметров, лучше для маленьких моделей). Эмбеддинги начинаются маленькими (стандартное отклонение 0,02), а числа синусоидального позиционного кодирования – порядка 1; чтобы токены не терялись на фоне позиций, эмбеддинги умножают на $\sqrt d$, как в исходной статье;
- `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` – ограничение нормы градиента: защищает обучение от редких огромных шагов;
- `AutoTokenizer.from_pretrained(name)`, `AutoModel.from_pretrained(name)`, `AutoModelForSequenceClassification.from_pretrained(name, num_labels=5)` – загрузка токенизатора, кодировщика и кодировщика с головой классификации;
- `tok(texts, padding=True, truncation=True, return_tensors='pt')` – пакет фраз в номера токенов одной длины с маской внимания `attention_mask`; `model(**batch).last_hidden_state[:, 0]` – векторы `[CLS]`; `model(**batch, labels=y).loss` – потери классификатора;
- `model.save_pretrained(папка)`, `tok.save_pretrained(папка)` – сохранить дообученную модель.

In [ ]:
# Ячейка подготовки: режим, проверка среды, данные, готовые блоки
MODE = 'легкий'          # 'легкий' – компьютер в аудитории; 'полный' – Colab с видеокартой
SKIP_TRAINING = False    # True – не обучать GPT, а загрузить готовую контрольную точку
USE_DRIVE = False        # True (только в Colab) – хранить контрольные точки на Google Диске

import csv
import json
import math
import platform
import re
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display
import matplotlib.pyplot as plt
from IPython.display import Image, display

assert MODE in ('легкий', 'полный'), "MODE должен быть 'легкий' или 'полный'"
IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python', platform.python_version(), '| PyTorch', torch.__version__, '| среда:', 'Google Colab' if IN_COLAB else 'свой компьютер', '| режим:', MODE)
if DEVICE == 'cuda':
    print('видеокарта:', torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} ГБ')
else:
    print('видеокарты нет, все считается на процессоре')
    if MODE == 'полный':
        print('ВНИМАНИЕ: полный режим рассчитан на видеокарту; на процессоре обучение GPT займет очень долго – включите SKIP_TRAINING или легкий режим')

def ensure(module, package, version):
    """Ставит библиотеку фиксированной версии, только если ее нет."""
    try:
        __import__(module)
    except ImportError:
        print(f'устанавливаю {package}=={version} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--retries', '1', '--timeout', '10', f'{package}=={version}'], check=False)

ensure('transformers', 'transformers', '4.51.3')
ensure('sklearn', 'scikit-learn', '1.5.2')

NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'
REPO_RAW = 'https://raw.githubusercontent.com/ExPl0i/AI-10-NLP-Language-models/main/data/'
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

def fetch(name, required=True):
    """Файл из папки data; если его нет – скачиваем из репозитория курса."""
    path = DATA_DIR / name
    if not path.exists():
        try:
            print(f'скачиваю {name} ...')
            urllib.request.urlretrieve(REPO_RAW + name, path)
        except Exception as error:
            path.unlink(missing_ok=True)
            if required:
                print(f'Не удалось скачать {name}:', error)
    return path

CKPT_DIR = Path('checkpoints')
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = Path('/content/drive/MyDrive/AI-10-NLP-Language-models')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

SETTINGS = {'легкий': dict(bpe='light', d_model=128, n_heads=4, d_ff=512, n_layers=4, ctx=64, batch=32, steps=1500, lr=3e-3),
            'полный': dict(bpe='full', d_model=384, n_heads=6, d_ff=1536, n_layers=6, ctx=256, batch=32, steps=2000, lr=5e-4)}
S = SETTINGS[MODE]
INTENTS = ['навигация', 'климат', 'музыка', 'звонок', 'состояние']

def expect(light, full):
    """Ожидаемое значение для выбранного режима."""
    return light if MODE == 'легкий' else full

In [ ]:
# Готовые блоки: токенизатор BPE из Т2.Б1.1 и блоки трансформера из Т2.Б1.3
PATTERN = r"[^\W\d_]+|\d+|\s+|[^\w\s]|_"
SPECIALS = ['<pad>', '<unk>', '<bos>', '<eos>']

def merge(tokens, pair, new_token):
    out, i = [], 0
    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == pair:
            out.append(new_token); i += 2
        else:
            out.append(tokens[i]); i += 1
    return out

class BPETokenizer:
    """Токенизатор из Т2.Б1.1: правила объединения в порядке обучения, кэш разбиений слов."""
    def __init__(self, path):
        data = json.loads(Path(path).read_text(encoding='utf-8'))
        self.merges = [tuple(m) for m in data['merges']]
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self.itos, seen = list(SPECIALS), set(SPECIALS)
        for t in data['alphabet'] + [a + b for a, b in self.merges]:
            if t not in seen:
                self.itos.append(t); seen.add(t)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.cache = {}

    def encode_word(self, word):
        if word not in self.cache:
            tokens = list(word)
            while len(tokens) > 1:
                best = min(zip(tokens, tokens[1:]), key=lambda p: self.ranks.get(p, math.inf))
                if best not in self.ranks:
                    break
                tokens = merge(tokens, best, best[0] + best[1])
            self.cache[word] = [self.stoi.get(t, 1) for t in tokens]
        return self.cache[word]

    def encode(self, text):
        return [i for w in re.findall(PATTERN, text) for i in self.encode_word(w)]

    def decode(self, ids):
        return ''.join(self.itos[i] for i in ids if i not in (0, 2, 3))

def attention(Q, K, V, mask=None):
    scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights

def causal_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))

def sinusoidal_pe(max_len, d_model):
    p = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    i2 = torch.arange(0, d_model, 2, dtype=torch.float32).unsqueeze(0)
    angle = p / (10000 ** (i2 / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(angle), torch.cos(angle)
    return pe

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.d_k = n_heads, d_model // n_heads
        self.q_proj, self.k_proj = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)
        self.v_proj, self.out_proj = nn.Linear(d_model, d_model), nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        split = lambda t: t.view(B, T, self.h, self.d_k).transpose(1, 2)
        out, _ = attention(split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x)), mask)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, D))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d_model), MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        return x + self.ff(self.ln2(x))

corpus = fetch('corpus.txt').read_text(encoding='utf-8')
bpe = BPETokenizer(fetch(f"bpe_{S['bpe']}.json"))
start = time.perf_counter()
tokens = torch.tensor(bpe.encode(corpus), dtype=torch.long)
n_train = int(len(tokens) * 0.95)
train_data, val_data = tokens[:n_train], tokens[n_train:]
print(f'словарь BPE {len(bpe.itos)}, корпус {len(corpus)} символов -> {len(tokens)} токенов за {time.perf_counter() - start:.0f} с',
      '  ожидается', expect('1000 и 4366018', '2000 и 3976118'))
val_chars = len(bpe.decode(val_data.tolist()))
print(f'обучающая часть {len(train_data)} токенов, проверочная {len(val_data)} токенов ({val_chars} символов)')

**Задание Г1. Свой маленький GPT.**

Допишите модель `GPT` и выборку пакетов:
- в `__init__` уже есть эмбеддинги токенов, позиционное кодирование, причинная маска (обе – буферы, они не обучаются), блоки из Т2.Б1.3, последняя нормализация и выходной слой с общими весами с эмбеддингами;
- `forward(idx, targets=None)`: эмбеддинги, умноженные на $\sqrt d$, плюс позиции первых `T` мест, все блоки с причинной маской `self.mask[:T, :T]`, последняя нормализация, выходной слой. Если переданы цели, вернуть еще потери `F.cross_entropy` по всем позициям, иначе `None`;
- `get_batch(data, batch, ctx, gen)`: выбрать `batch` случайных начал отрезков, вход `x` – отрезки длины `ctx`, цель `y` – те же отрезки, сдвинутые на один токен вправо (п. 2 теории).

Затем допишите четыре строки шага обучения. Готовая ячейка обучает модель, раз в 250 шагов печатает потери на проверочной части и сохраняет контрольную точку; если точка уже есть, обучение продолжается с нее.

*Проверка.* Для данных `[0, 1, ..., 9]`, отрезков длины 3 и генератора с зерном 0 вход и цель отличаются сдвигом на 1. Параметров 921 344 в легком режиме и 11 415 552 в полном. Потери нового GPT около $\ln V$: 6,91 при словаре 1000 и 7,60 при словаре 2000.

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, d_model, n_heads, d_ff, n_layers, ctx):
        super().__init__()
        self.ctx, self.d_model = ctx, d_model
        self.tok = nn.Embedding(vocab, d_model)
        self.register_buffer('pe', sinusoidal_pe(ctx, d_model), persistent=False)
        self.register_buffer('mask', causal_mask(ctx), persistent=False)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.tok.weight                         # общие веса, как в GPT-2
        for m in self.modules():                                   # начальные веса, как в GPT-2: нормальные со стандартным отклонением 0,02
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = ...                                     # <-- эмбеддинги токенов, умноженные на sqrt(d), + позиции 0..T-1
        for block in self.blocks:
            x = ...                                 # <-- блок с причинной маской
        logits = ...                                # <-- последняя нормализация и выходной слой: (B, T, V)
        if targets is None:
            return logits, None
        loss = ...                                  # <-- потери по всем позициям
        return logits, loss

def get_batch(data, batch, ctx, gen):
    starts = torch.randint(0, len(data) - ctx - 1, (batch,), generator=gen)
    x = ...                                         # <-- отрезки длины ctx
    y = ...                                         # <-- те же отрезки, сдвинутые на 1
    return x, y


xb, yb = get_batch(torch.arange(10), 2, 3, torch.Generator().manual_seed(0))
print(xb.tolist(), yb.tolist(), '  ожидается: каждая строка y – строка x плюс 1, числа от 0 до 9')
model = GPT(len(bpe.itos), S['d_model'], S['n_heads'], S['d_ff'], S['n_layers'], S['ctx']).to(DEVICE)
print('параметров:', sum(p.numel() for p in model.parameters()), '  ожидается', expect(921344, 11415552))
with torch.no_grad():
    x0, y0 = get_batch(train_data, 8, S['ctx'], torch.Generator().manual_seed(1))
    _, loss0 = model(x0.to(DEVICE), y0.to(DEVICE))
print(f'потери до обучения {loss0.item():.3f}', f'  ожидается около ln V = {math.log(len(bpe.itos)):.3f}')

In [ ]:
# Обучение GPT с контрольными точками (легкий режим – несколько минут на процессоре, полный – 5–10 минут на видеокарте)
CKPT = CKPT_DIR / f"gpt_{S['bpe']}.pt"

def estimate_loss(model, data, n_batches=20, seed=99):
    """Средние потери на n_batches случайных пакетах (одни и те же при одинаковом seed)."""
    model.eval()
    gen = torch.Generator().manual_seed(seed)
    losses = []
    with torch.no_grad():
        for _ in range(n_batches):
            x, y = get_batch(data, S['batch'], S['ctx'], gen)
            losses.append(model(x.to(DEVICE), y.to(DEVICE))[1].item())
    model.train()
    return float(np.mean(losses))

def load_ready_checkpoint(model):
    """Готовая контрольная точка: gpt_light.npz из репозитория или gpt_full.pt от преподавателя."""
    if S['bpe'] == 'light':
        state = {k: torch.from_numpy(v) for k, v in np.load(fetch('gpt_light.npz')).items()}
    else:
        state = torch.load(fetch('gpt_full.pt'), map_location='cpu')['model']
    model.load_state_dict(state)

history = []
if SKIP_TRAINING:
    load_ready_checkpoint(model)
    print('загружена готовая контрольная точка')
else:
    optimizer = torch.optim.Adam(model.parameters(), lr=S['lr'])
    step0 = 0
    if CKPT.exists():                                              # продолжить с сохраненного места
        saved = torch.load(CKPT, map_location=DEVICE)
        model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
        step0, history = saved['step'], saved['history']
        print(f'продолжаю с шага {step0}')
    gen = torch.Generator().manual_seed(step0)
    start = time.perf_counter()
    for step in range(step0 + 1, S['steps'] + 1):
        x, y = get_batch(train_data, S['batch'], S['ctx'], gen)
        logits, loss = model(x.to(DEVICE), y.to(DEVICE))
        ...                                         # <-- обнулить градиенты
        ...                                         # <-- обратный проход
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        ...                                         # <-- шаг оптимизатора
        if step % 250 == 0 or step == step0 + 1:
            val = estimate_loss(model, val_data)
            history.append((step, loss.item(), val))
            print(f'шаг {step:>5}: потери {loss.item():.3f}, на проверке {val:.3f}, перплексия {math.exp(val):.1f}, {time.perf_counter() - start:.0f} с')
            torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'step': step, 'history': history}, CKPT)
print('  ожидается: потери на проверке падают с', expect('6.3 до примерно 3.1', '7.6 и ниже; если потери на проверке начали расти – это переобучение'))

**Вопрос Г1.** а) Почему из одного отрезка длины 64 получается 64 обучающих примера, а не один? б) Зачем выходной слой делит веса с эмбеддингами? Сколько параметров это сэкономило в вашем режиме? в) Почему потери на проверочной части выше, чем на обучающей, хотя модель ни разу не видела проверочный текст и, значит, не могла его запомнить?

*Ваш ответ:*

**Задание Г2. Генерация, перплексия и сравнение с символьной моделью.**

- `generate(model, idx, n_new, temperature=1.0)`: `n_new` раз взять последние `model.ctx` токенов, получить оценки последней позиции, разделить на температуру, softmax, выбрать токен `torch.multinomial` и дописать его в конец (п. 4 теории).
- `perplexity(loss)` – $e^{L}$; `bits_per_char(loss, n_tokens, n_chars)` – формула из п. 3 теории.

Готовая ячейка генерирует продолжения начала фразы при разных температурах, считает перплексию на проверочной части и сравнивает ее с двумя простыми моделями: «равновероятный выбор» (перплексия $V$) и «частота токенов без контекста». Затем она обучает символьную рекуррентную модель LSTM, как в Теме 1, столько же времени и сравнивает биты на символ.

*Проверка.* `perplexity(math.log(1000))` = 1000; для потерь 2 нат на токен, 1000 токенов и 1700 символов `bits_per_char` = 1,697. Генерация из модели, которая всегда предсказывает токен 7, дописывает только семерки.

In [ ]:
@torch.no_grad()
def generate(model, idx, n_new, temperature=1.0):
    model.eval()
    for _ in range(n_new):
        context = ...                               # <-- последние ctx токенов
        logits, _ = model(context)
        logits = ...                                # <-- оценки последней позиции, деленные на температуру
        probs = ...                                 # <-- распределение следующего токена
        next_id = ...                               # <-- случайный токен по распределению
        idx = ...                                   # <-- дописать в конец
    model.train()
    return idx

def perplexity(loss):
    return ...                                      # <-- e в степени потерь

def bits_per_char(loss, n_tokens, n_chars):
    return ...                                      # <-- биты на символ


class Always7(nn.Module):
    ctx = 4
    def forward(self, idx, targets=None):
        logits = torch.full((*idx.shape, 10), -1e9); logits[..., 7] = 0
        return logits, None

print(generate(Always7(), torch.tensor([[1, 2]]), 5).tolist(), '  ожидается [[1, 2, 7, 7, 7, 7, 7]]')
print(perplexity(math.log(1000)), f'{bits_per_char(2.0, 1000, 1700):.3f}', '  ожидается 1000.0 (с точностью до округления) и 1.697')

In [ ]:
# Готовая ячейка: тексты, перплексия и простые модели для сравнения
torch.manual_seed(0)
prompt = 'Вечером, когда солнце село за лес,'
for temperature in (0.5, 1.0, 1.5):
    ids = generate(model, torch.tensor([bpe.encode(prompt)], device=DEVICE), 60, temperature)
    print(f'температура {temperature}:', bpe.decode(ids[0].tolist()).replace('\n', ' '))
    print()

val_loss = estimate_loss(model, val_data, n_batches=50)
counts = torch.bincount(train_data, minlength=len(bpe.itos)).float() + 1
p_unigram = counts / counts.sum()
unigram_loss = -torch.log(p_unigram[val_data]).mean().item()
print(f'{"модель":<28}{"потери":>8}{"перплексия":>12}{"бит на символ":>15}')
for name, loss in [('равновероятный выбор', math.log(len(bpe.itos))), ('частота токенов', unigram_loss), ('наш GPT', val_loss)]:
    print(f'{name:<28}{loss:>8.3f}{perplexity(loss):>12.1f}{bits_per_char(loss, len(val_data), val_chars):>15.3f}')
print('  ожидается: у GPT перплексия', expect('около 22', 'заметно меньше, чем у простых моделей'), '– в разы меньше, чем у простых моделей')

In [ ]:
# Готовая ячейка: символьная LSTM, как в Теме 1, на том же корпусе и столько же времени
chars = sorted(set(corpus))
c2i = {c: i for i, c in enumerate(chars)}
char_data = torch.tensor([c2i[c] for c in corpus], dtype=torch.long)
n_char_train = len(char_data) - val_chars                          # проверка – тот же конец корпуса

class CharLSTM(nn.Module):
    def __init__(self, vocab, emb=64, hidden=256):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, vocab)

    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        return self.out(h)

lstm = CharLSTM(len(chars)).to(DEVICE)
opt = torch.optim.Adam(lstm.parameters(), lr=3e-3)
gen = torch.Generator().manual_seed(0)
budget = expect(120, 300)                                          # секунд на обучение
start, steps = time.perf_counter(), 0
while time.perf_counter() - start < budget:
    x, y = get_batch(char_data[:n_char_train], 32, 128, gen)
    loss = F.cross_entropy(lstm(x.to(DEVICE)).reshape(-1, len(chars)), y.to(DEVICE).reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); steps += 1
lstm.eval()
with torch.no_grad():
    gv = torch.Generator().manual_seed(99)
    lstm_loss = np.mean([F.cross_entropy(lstm(x.to(DEVICE)).reshape(-1, len(chars)), y.to(DEVICE).reshape(-1)).item()
                         for x, y in (get_batch(char_data[n_char_train:], 32, 128, gv) for _ in range(30))])
print(f'символьная LSTM: {steps} шагов за {budget} с, {sum(p.numel() for p in lstm.parameters())} параметров, '
      f'бит на символ {lstm_loss / math.log(2):.3f}')
print(f'наш GPT: бит на символ {bits_per_char(val_loss, len(val_data), val_chars):.3f}')
print('  ожидается: обе модели намного лучше частотной модели символов (около 4.8 бита на символ);',
      'кто из двух лучше, зависит от времени обучения и компьютера')

**Вопрос Г2.** а) Как меняется текст при температуре 0,5, 1 и 1,5? Объясните по формуле softmax. б) Что означает перплексия вашего GPT простыми словами? Почему перплексию GPT на словаре 1000 нельзя напрямую сравнить с перплексией символьной LSTM, а биты на символ – можно? в) Текст GPT похож на русский, но смысла в нем мало. Чего не хватает модели по сравнению с большими языковыми моделями?

*Ваш ответ:*

**Задание Г3. rubert-tiny2: векторы фраз и смысловой поиск.**

Готовая ячейка загружает фразы водителя и модель rubert-tiny2. Допишите:
- `embed(texts, batch=64)` – векторы фраз по п. 7 теории: токенизатор с `padding=True, truncation=True, max_length=64, return_tensors='pt'`, выход модели `last_hidden_state`, вектор токена `[CLS]` (позиция 0), нормировка `F.normalize`. Возвращает тензор `(N, 312)`;
- `search(query, base_texts, base_vectors, k=5)` – `k` фраз базы с наибольшим косинусом к запросу, список пар «фраза, косинус».

*Проверка.* Векторы двух фраз имеют форму `(2, 312)` и длину 1. Затем готовая ячейка ищет по базе обучающих фраз ответы на запросы, которых в базе нет, и смотрит, какому намерению принадлежат найденные фразы.

In [ ]:
# Готовая ячейка: фразы водителя и модель rubert-tiny2
def read_intents(path):
    with open(path, encoding='utf-8') as f:
        rows = [(r['text'].strip(), r['intent'].strip()) for r in csv.DictReader(f)]
    return [(t, i) for t, i in rows if t and i in INTENTS]

own = Path('intents.csv')
train_rows = read_intents(own) if own.exists() else read_intents(fetch('intents_reserve.csv'))
test_rows = read_intents(fetch('intents_test.csv'))
print('обучающие фразы:', 'ваш файл intents.csv' if own.exists() else 'запасной набор', len(train_rows), dict(Counter(i for _, i in train_rows)))
print('тестовые фразы:', len(test_rows))

from transformers import AutoModel, AutoTokenizer
BERT_NAME = 'cointegrated/rubert-tiny2'
local = Path('models') / 'rubert-tiny2'                            # для аудитории без интернета
bert_tok = AutoTokenizer.from_pretrained(str(local) if local.exists() else BERT_NAME)
bert = AutoModel.from_pretrained(str(local) if local.exists() else BERT_NAME).eval()
print('rubert-tiny2:', sum(p.numel() for p in bert.parameters()), 'параметров, словарь', bert_tok.vocab_size)
print('токены фразы:', bert_tok.tokenize('Включи обогрев сидений, мне холодно'))

In [ ]:
@torch.no_grad()
def embed(texts, batch=64):
    vectors = []
    for i in range(0, len(texts), batch):
        enc = ...                                   # <-- пакет фраз в номера токенов
        hidden = ...                                # <-- выходы кодировщика: (пакет, длина, 312)
        vectors.append(...)                         # <-- вектор [CLS], нормированный к длине 1
    return torch.cat(vectors)

def search(query, base_texts, base_vectors, k=5):
    q = ...                                         # <-- вектор запроса
    sims = ...                                      # <-- косинусы (векторы уже нормированы)
    best = ...                                      # <-- номера k наибольших
    return [(base_texts[i], round(sims[i].item(), 3)) for i in best]


v = embed(['мне холодно', 'включи музыку'])
print(tuple(v.shape), v.norm(dim=1).tolist(), '  ожидается (2, 312) [1.0, 1.0]')

In [ ]:
# Готовая ячейка: смысловой поиск по обучающим фразам
base_texts = [t for t, _ in train_rows]
base_intent = {t: i for t, i in train_rows}
start = time.perf_counter()
base_vectors = embed(base_texts)
print(f'векторы {len(base_texts)} фраз за {time.perf_counter() - start:.0f} с')
for query in ['мне холодно', 'стекла запотели', 'хочу домой', 'наберешь Сашу?', 'давай что-нибудь повеселее', 'мигает значок на панели']:
    found = search(query, base_texts, base_vectors, k=3)
    print(f'«{query}» ->', '; '.join(f'{t} [{base_intent[t]}, {s}]' for t, s in found))
print('  ожидается: у большинства запросов найденные фразы того же намерения, хотя общих слов может не быть')

**Вопрос Г3.** а) Найдите запрос, для которого найденные фразы правильного намерения не имеют с ним ни одного общего слова. Почему поиск все равно сработал? б) Чем вектор фразы rubert-tiny2 лучше среднего векторов слов из Т2.Б1.2? Приведите пару фраз, которые среднее векторов слов не различит. в) Почему вектор `[CLS]` вообще описывает всю фразу, хотя это всего один служебный токен?

*Ваш ответ:*

**Задание Г4. Дообучение rubert-tiny2 определять намерение.**

Готовая ячейка делит обучающие фразы на обучение и проверку (90 и 10 %) и загружает rubert-tiny2 с головой классификации на 5 намерений. Допишите `train_classifier`: в каждой эпохе перемешать примеры, пройти пакетами по 32: токенизировать пакет, получить потери `model(**enc, labels=y).loss`, обнулить градиенты, обратный проход, шаг оптимизатора. После эпохи напечатать точность на проверке функцией `accuracy` (уже написана).

*Проверка.* До дообучения точность на проверке около 0,2 (голова случайная, это угадывание одного из пяти). После 3 эпох на процессоре (около минуты) точность на проверке близка к 1. Затем готовая ячейка сохраняет классификатор в папку `intent_classifier` – он понадобится на Т2.Б2.5.

In [ ]:
# Готовая ячейка: разбиение и модель с головой классификации
from transformers import AutoModelForSequenceClassification
label2id = {name: i for i, name in enumerate(INTENTS)}
rng = np.random.default_rng(0)
order = rng.permutation(len(train_rows))
n_val = len(train_rows) // 10
val_rows = [train_rows[i] for i in order[:n_val]]
fit_rows = [train_rows[i] for i in order[n_val:]]
print(f'обучение {len(fit_rows)}, проверка {len(val_rows)}, тест {len(test_rows)}')

torch.manual_seed(0)
clf = AutoModelForSequenceClassification.from_pretrained(str(local) if local.exists() else BERT_NAME, num_labels=len(INTENTS),
                                                         id2label=dict(enumerate(INTENTS)), label2id=label2id)

@torch.no_grad()
def predict(model, texts, batch=64):
    model.eval()
    out = []
    for i in range(0, len(texts), batch):
        enc = bert_tok(texts[i:i + batch], padding=True, truncation=True, max_length=64, return_tensors='pt')
        out += model(**enc).logits.argmax(-1).tolist()
    model.train()
    return out

def accuracy(model, rows):
    pred = predict(model, [t for t, _ in rows])
    return float(np.mean([p == label2id[i] for p, (_, i) in zip(pred, rows)]))

print(f'точность на проверке до дообучения: {accuracy(clf, val_rows):.2f}', '  ожидается около 0.2')

In [ ]:
def train_classifier(model, rows, epochs=3, lr=5e-5, batch=32, seed=0):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    model.train()
    for epoch in range(epochs):
        start = time.perf_counter()
        order = rng.permutation(len(rows))
        losses = []
        for b in range(0, len(order), batch):
            part = [rows[i] for i in order[b:b + batch]]
            enc = ...                               # <-- пакет фраз в токены
            y = torch.tensor([label2id[i] for _, i in part])
            loss = ...                              # <-- потери классификатора
            ...                                     # <-- обнулить градиенты
            ...                                     # <-- обратный проход
            ...                                     # <-- шаг оптимизатора
            losses.append(loss.item())
        print(f'эпоха {epoch + 1}: потери {np.mean(losses):.3f}, точность на проверке {accuracy(model, val_rows):.3f}, '
              f'{time.perf_counter() - start:.0f} с')
    return model


clf = train_classifier(clf, fit_rows)
print('  ожидается: потери падают, точность на проверке близка к 1')
clf.save_pretrained('intent_classifier')
bert_tok.save_pretrained('intent_classifier')
print('классификатор сохранен в папку intent_classifier')

**Вопрос Г4.** а) Почему для классификации намерений выбран кодировщик, а не модель вроде нашего GPT? б) Почему шаг обучения такой маленький ($5 \cdot 10^{-5}$), а эпох всего три? Что случится при шаге $10^{-2}$? в) Точность на проверке близка к 1. Почему этого недостаточно, чтобы считать классификатор хорошим?

*Ваш ответ:*

**Задание Г5. Метрики классификатора и разбор ошибок.**

Допишите `confusion(true, pred, n)` – матрицу ошибок $n \times n$ (строка – истинный класс, столбец – предсказанный) и `f1_per_class(C)` – F-меры классов по п. 9 теории (при нулевом знаменателе F-мера равна 0).

*Проверка.* Для истинных `[0, 0, 1, 1, 2]` и предсказанных `[0, 1, 1, 1, 0]` матрица `[[1, 1, 0], [0, 2, 0], [1, 0, 0]]`, F-меры `[0.5, 0.8, 0.0]`, макро-F-мера 0,433; на тестовых фразах ваши числа совпадают с `sklearn`. Затем готовая ячейка рисует матрицу ошибок на тестовых живых фразах, сравнивает дообученный rubert-tiny2 с простой моделью из 9 класса (TF-IDF по кусочкам слов и логистическая регрессия) и печатает ошибки.

In [ ]:
def confusion(true, pred, n):
    C = np.zeros((n, n), dtype=int)
    for t, p in zip(true, pred):
        ...                                         # <-- строка – истинный класс, столбец – предсказанный
    return C

def f1_per_class(C):
    tp = ...                                        # <-- верно предсказанные каждого класса
    precision = ...                                 # <-- точность: делим на суммы столбцов
    recall = ...                                    # <-- полнота: делим на суммы строк
    s = precision + recall
    return ...                                      # <-- F-мера


C_demo = confusion([0, 0, 1, 1, 2], [0, 1, 1, 1, 0], 3)
print(C_demo.tolist(), f1_per_class(C_demo).round(3).tolist(), f'{f1_per_class(C_demo).mean():.3f}',
      '  ожидается [[1, 1, 0], [0, 2, 0], [1, 0, 0]] [0.5, 0.8, 0.0] 0.433')

In [ ]:
# Готовая ячейка: тестовые живые фразы, сравнение с TF-IDF и разбор ошибок
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

test_texts = [t for t, _ in test_rows]
y_true = [label2id[i] for _, i in test_rows]
y_bert = predict(clf, test_texts)
C = confusion(y_true, y_bert, len(INTENTS))
f1 = f1_per_class(C)
print('F-меры классов:', dict(zip(INTENTS, f1.round(3))), f'| макро-F-мера {f1.mean():.3f}')
print('совпадает со sklearn:', np.isclose(f1.mean(), f1_score(y_true, y_bert, average='macro')), '  ожидается True')

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_fit = tfidf.fit_transform([t for t, _ in train_rows])
logreg = LogisticRegression(max_iter=2000, C=10).fit(X_fit, [label2id[i] for _, i in train_rows])
y_tfidf = logreg.predict(tfidf.transform(test_texts)).tolist()
print(f'макро-F-мера на тестовых фразах: rubert-tiny2 {f1.mean():.3f}, TF-IDF и логистическая регрессия '
      f'{f1_per_class(confusion(y_true, y_tfidf, len(INTENTS))).mean():.3f}')
print('  ожидается: у TF-IDF на запасном наборе около 0.819; дообученный кодировщик обычно лучше')

fig, ax = plt.subplots(figsize=(5, 4.2))
ax.imshow(C, cmap='Blues')
ax.set_xticks(range(5), INTENTS, rotation=40, ha='right'); ax.set_yticks(range(5), INTENTS)
for i in range(5):
    for j in range(5):
        ax.text(j, i, C[i, j], ha='center', va='center', color='white' if C[i, j] > C.max() / 2 else NAVY)
ax.set_xlabel('предсказано'); ax.set_ylabel('на самом деле'); ax.set_title('rubert-tiny2 на живых фразах')
fig.tight_layout(); fig.savefig('confusion.png', dpi=110); plt.close(fig)
display(Image('confusion.png'))
print('ошибки:')
for t, yt, yp in zip(test_texts, y_true, y_bert):
    if yt != yp:
        print(f'  «{t}»: на самом деле {INTENTS[yt]}, модель – {INTENTS[yp]}')

**Вопрос Г5.** а) Какие намерения модель путает чаще всего и почему именно их? Приведите две ошибки и объясните, чем фраза сбивает модель. б) Зачем считать макро-F-меру, если есть доля верных ответов? В каком случае они сильно разойдутся? в) Как улучшить классификатор, не меняя модель? Предложите два способа.

*Ваш ответ:*

### Задание повышенной сложности (к части 4Г)

1. **Замороженный кодировщик.** Дообучите новый классификатор, заморозив все веса кодировщика (`for p in clf.bert.parameters(): p.requires_grad = False`) и обучая только голову, с шагом $10^{-3}$ и 10 эпохами. Сравните макро-F-меру на тестовых фразах и время обучения с размороженным кодировщиком. Объясните результат через трансферное обучение из Темы 1.
2. **Шестое намерение без переобучения всей модели.** Добавьте намерение «окна и двери» («открой окно», «закрой люк», «заблокируй двери»): соберите 30 фраз и классифицируйте по ближайшему среднему вектору класса (векторы `embed` из Г3) – без обучения вообще. Как это работает на остальных пяти намерениях по сравнению с дообученной моделью?
3. **GPT по кусочку слова и по символу.** Обучите тот же GPT на символах вместо токенов BPE (словарь – символы корпуса) с тем же числом шагов и сравните биты на символ. Почему BPE выигрывает при одинаковом окне контекста?

In [ ]:
# Место для задания повышенной сложности